# OrbitsWar — Fine-Tune 4v4 Models vs Bots

Warm-starts the 4v4 GBC and GBR from the base models saved by `train_models_kaggle.ipynb`,
then fits additional trees on exhaustive C(N,3) bot-match data.

| Model | Script | Input | Output |
|---|---|---|---|
| 4v4 GBC (classifier) | `heuristic_4v4/train_4v4_vs_bots.py` | `heuristic_4v4/model.pkl` | `heuristic_4v4/value_fn_4v4.py` |
| 4v4 GBR (regressor)  | `regressor_4v4/train_vs_bots.py`     | `regressor_4v4/model.pkl` | `regressor_4v4/value_fn_4v4.py` |

**Prerequisite:** Run `train_models_kaggle.ipynb` first (or attach its output dataset) so `model.pkl` files exist.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PASSES           = 1    # full C(N,3) cycles per model (increase for more coverage)
SNAPSHOT_EVERY   = 5    # classifier: collect a record every N steps

WORK_DIR = "/kaggle/working/OrbitsWar"

import os

def find_dataset_path():
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    for root, dirs, files in os.walk(base):
        dirs.sort()
        if "main.py" in files and "records" in dirs:
            print(f"Found OrbitsWar dataset at: {root}")
            return root
    print("Did not find main.py + records/ anywhere under /kaggle/input/")
    return None

INPUT_DIR = find_dataset_path()
if INPUT_DIR is None:
    raise RuntimeError(
        "Could not find OrbitsWar dataset. Attach the dataset to this notebook."
    )

print(f"Input:   {INPUT_DIR}")
print(f"WorkDir: {WORK_DIR}")
print(f"Passes:  {PASSES}")

In [ ]:
# ── Copy dataset to writable working directory ─────────────────────────────────
import shutil, sys

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

shutil.copytree(INPUT_DIR, WORK_DIR)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

print(f"Copied to {WORK_DIR}")
print("Contents:", sorted(os.listdir(WORK_DIR)))

In [ ]:
!pip install -q kaggle-environments scikit-learn joblib
print("Dependencies ready.")

In [ ]:
# ── Verify prerequisites ───────────────────────────────────────────────────────
import glob

records = glob.glob("records/*.json")
print(f"Replay files : {len(records)}")

enemies = glob.glob("enemy agents/*/main.py") + glob.glob("enemy agents/my/*.py")
enemies = [e for e in enemies if not e.endswith("__init__.py")]
print(f"Enemy agents : {len(enemies)}")

for pkl, label in [
    ("heuristic_4v4/model.pkl",  "4v4 classifier base model"),
    ("regressor_4v4/model.pkl",  "4v4 regressor  base model"),
]:
    status = "[OK]" if os.path.exists(pkl) else "[MISSING — will train from scratch]"
    print(f"{status}  {pkl}  ({label})")

---
## Training Loop — Regressor then Classifier (per pass)
Each pass: 4v4 **Regressor** trains + saves `model.pkl` → 4v4 **Classifier** warm-starts from that save.  
`PASSES` controls how many full C(N,3) sweeps each model runs.

In [ ]:
import subprocess

for pass_num in range(1, PASSES + 1):
    print(f"\n{'='*60}")
    print(f"  Pass {pass_num}/{PASSES}")
    print(f"{'='*60}")

    print("\n--- 4v4 Regressor ---")
    subprocess.run(
        ["python", "regressor_4v4/train_vs_bots.py", "--passes", "1"],
        check=True
    )

    print("\n--- 4v4 Classifier ---")
    subprocess.run(
        ["python", "heuristic_4v4/train_4v4_vs_bots.py",
         "--passes", "1",
         "--snapshot-every", str(SNAPSHOT_EVERY)],
        check=True
    )

    print(f"\n  Pass {pass_num}/{PASSES} complete.")

In [ ]:
for p in ["regressor_4v4/value_fn_4v4.py", "heuristic_4v4/value_fn_4v4.py"]:
    tag = f"[OK]  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else "[FAIL]"
    print(f"{tag}  {p}")

---
## Output
Fine-tuned value functions are copied to `/kaggle/working/` for download.

In [ ]:
import shutil

outputs = [
    ("heuristic_4v4/value_fn_4v4.py", "heuristic_4v4_value_fn_4v4_bots.py"),
    ("regressor_4v4/value_fn_4v4.py", "regressor_4v4_value_fn_4v4_bots.py"),
]

print("Output files:")
print("-" * 60)
all_ok = True
for src, dest_name in outputs:
    if os.path.exists(src):
        sz  = os.path.getsize(src) / 1024
        dst = f"/kaggle/working/{dest_name}"
        shutil.copy(src, dst)
        print(f"  [OK]   {src:<45} {sz:>7.1f} KB")
    else:
        print(f"  [MISS] {src}")
        all_ok = False

print("-" * 60)
if all_ok:
    print("Both 4v4 models fine-tuned. Download from the Output panel.")
    print()
    print("Local placement after download:")
    print("  heuristic_4v4_value_fn_4v4_bots.py  -> heuristic_4v4/value_fn_4v4.py")
    print("  regressor_4v4_value_fn_4v4_bots.py  -> regressor_4v4/value_fn_4v4.py")
else:
    print("Some models failed — check cell output above.")